## Filtering Precipitation Data ##

This notebook works to filter precipitation data for quality filtering as well as space filtering. It does not use shapefiles, and it does not filter out urban areas. Rather, it just looks at entire midwest region as a whole. 

It filters for: 
* precipitation quality index
* probability liquid precipitation

It also works to: 
* implement grid size
* converts half hourly to hourly 

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

## loading data

In [2]:
#precipitation data from PPS 
precip_df_midwest = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_april_precip.parquet', 
                                    engine='pyarrow')

In [3]:
precip_df_midwest

,latitude,longitude,precipitation,randomError,probabilityLiquidPrecipitation,precipitationQualityIndex,date,time_
0,37.05,-103.950000,0.0,0.24,100,0.859,2024-04-11,11:00:00
1,37.05,-103.850000,0.0,0.24,100,0.893,2024-04-11,11:00:00
2,37.05,-103.750000,0.0,0.24,100,0.893,2024-04-11,11:00:00
3,37.05,-103.649994,0.0,0.24,100,0.893,2024-04-11,11:00:00
4,37.05,-103.549995,0.0,0.24,100,0.893,2024-04-11,11:00:00
...,...,...,...,...,...,...,...,...
41471995,48.95,-80.450000,0.0,0.24,100,0.784,2024-04-13,23:00:00
41471996,48.95,-80.350000,0.0,0.24,100,0.784,2024-04-13,23:00:00
41471997,48.95,-80.250000,0.0,0.24,100,0.784,2024-04-13,23:00:00
41471998,48.95,-80.150000,0.0,0.24,100,0.784,2024-04-13,23:00:00


In [4]:
precip_df_midwest['precipitation'].describe()

count    4.147200e+07
mean     2.726379e-01
std      1.290844e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.911000e+01
Name: precipitation, dtype: float64

We do not need to filter over the midwest here because it is done when we download PPS data 

## Filtering out unsatisfactory data 
filtering for:
* probability liquid precipitation
* random error
* precipitation quality index 

In [5]:
precip_df_filtered = precip_df_midwest[
    (precip_df_midwest['precipitationQualityIndex'] > 0.6) &
    (precip_df_midwest['probabilityLiquidPrecipitation'] > 90)
]

In [6]:
precip_df_filtered= precip_df_filtered.drop(columns={'randomError', 'probabilityLiquidPrecipitation', 'precipitationQualityIndex'})

In [7]:
precip_df_filtered['precipitation'].describe()

count    3.888593e+07
mean     2.734289e-01
std      1.294861e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.911000e+01
Name: precipitation, dtype: float64

#gridding 
grid_size = 0.1 
precip_df_filtered['grid_lat'] = (precip_df_filtered['latitude']//grid_size) * grid_size 
precip_df_filtered['grid_lon'] = (precip_df_filtered['longitude']//grid_size) * grid_size 

#precipitation column is originally given as precipitation rate of mm/hr for every 30 mins. multiply by 0.5 to get mm every 30 mins 
precip_df_filtered['precipitation'] = precip_df_filtered['precipitation']*0.5
precip_df_filtered = precip_df_filtered.rename(columns = {'precipitation': 'hh_precip'}) #rename precip column to half hourly precip 

#need to add an hour column for grouping by hour, first need to add datetime 
precip_df_filtered['datetime'] = pd.to_datetime(precip_df_filtered['date'] + ' ' + precip_df_filtered['time_'])
precip_df_filtered['hour'] = precip_df_filtered['datetime'].dt.floor('h')

#want hourly measurements instead of half-hourly 
precip_df_filtered = precip_df_filtered.groupby(
    ['grid_lat', 'grid_lon', 'date', 'hour'], as_index=False)['hh_precip'].sum().rename(columns={'hh_precip': 'h_precip'})

precip_df_filtered = precip_df_filtered.drop(columns={'date'})

precip_df_filtered['h_precip'].describe()

In [14]:
precip_df_filtered.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/pps_HH_04_midwest_filtered.parquet', engine='pyarrow')

### creating a function that processes a precipitation dataset

4/8/26 -- getting rid of random error threshold because it scales with precipitation amounts, so higher precipitation, the higher the error is. So it basically makes our precipitation only have low precipitation as opposed to high precpitation. :(

In [2]:
def process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90):
    precip_df = pd.read_parquet(precip_data_path, engine='pyarrow')#reading precip data

    #filtering for quality
    precip_df = precip_df[
        (precip_df['precipitationQualityIndex'] > quality_threshold) &
        (precip_df['probabilityLiquidPrecipitation'] > prob_threshold)]

    #gridding datetime, time as strings
    precip_df['grid_lat'] = (precip_df['latitude'] // 0.1) * 0.1 + 0.05
    precip_df['grid_lon'] = (precip_df['longitude'] // 0.1) * 0.1 + 0.05
    precip_df['hh_precip'] = precip_df['precipitation'] * 0.5
    precip_df['datetime'] = pd.to_datetime(
    precip_df['date'].astype(str) + ' ' + precip_df['time_'].astype(str))
    precip_df['hour'] = precip_df['datetime'].dt.floor('h')

    #convert half-hourly measurements to hourly
    result = precip_df.groupby(
        ['grid_lat', 'grid_lon', 'date', 'hour'], as_index=False)['hh_precip'].sum().rename(columns={'hh_precip': 'h_precip_mm'})
    
    result.to_parquet(output_path, engine='pyarrow')
    print(f"filtered precip data saved to: {output_path}")

Testing the code 

In [3]:
precip_data_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_may_precip.parquet'
output_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_05_midwest_filtered.parquet'

process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90)

filtered precip data saved to: /Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_05_midwest_filtered.parquet


In [4]:
precip_data_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_june_precip.parquet'
output_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_06_midwest_filtered.parquet'

process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90)

filtered precip data saved to: /Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_06_midwest_filtered.parquet


In [5]:
precip_data_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_mar_precip.parquet'
output_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_03_midwest_filtered.parquet'

process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90)

filtered precip data saved to: /Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_03_midwest_filtered.parquet


In [6]:
precip_data_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_july_precip.parquet'
output_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_07_midwest_filtered.parquet'

process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90)

filtered precip data saved to: /Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_07_midwest_filtered.parquet


In [7]:
precip_data_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/pre_filter/pps_HH_april_precip.parquet'
output_path= '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_04_midwest_filtered.parquet'

process_precip_data(precip_data_path, output_path, grid_size=0.1, quality_threshold=0.6, prob_threshold=90)

filtered precip data saved to: /Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_04_midwest_filtered.parquet


In [2]:
march_precip_df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_03_midwest_filtered.parquet')
april_precip_df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_04_midwest_filtered.parquet')
may_precip_df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_05_midwest_filtered.parquet')
june_precip_df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_06_midwest_filtered.parquet')
july_precip_df = pd.read_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/compiled/filtered/pps_H_07_midwest_filtered.parquet')


In [3]:
precip_df = pd.concat([march_precip_df, april_precip_df, may_precip_df, june_precip_df, july_precip_df], ignore_index=True)
precip_df = precip_df.sort_values(['grid_lat', 'grid_lon', 'hour']).reset_index(drop=True)
precip_df['date'] = pd.to_datetime(precip_df['date']).dt.date
precip_df['hour'] = pd.to_datetime(precip_df['hour'])


# saving output

In [4]:
output_file = '/Volumes/share-sat2/SAT/TEMPO/Toireasa/precip_data/03-07_0.1x0.1gridded_centrally_midwest_precip.parquet'
precip_df.to_parquet(output_file, engine='pyarrow')

In [5]:
testing_df= pd.read_parquet(output_file, engine='pyarrow')

In [6]:
testing_df.sort_values('date')

,grid_lat,grid_lon,date,hour,h_precip_mm
0,37.05,-103.95,2024-03-01,2024-03-01 00:00:00,0.000
70646112,45.35,-95.85,2024-03-01,2024-03-01 21:00:00,0.015
70646113,45.35,-95.85,2024-03-01,2024-03-01 22:00:00,0.100
70646114,45.35,-95.85,2024-03-01,2024-03-01 23:00:00,0.225
35053177,41.15,-82.15,2024-03-01,2024-03-01 23:00:00,0.000
...,...,...,...,...,...
80642326,46.55,-103.85,2024-07-31,2024-07-31 21:00:00,0.020
80642325,46.55,-103.85,2024-07-31,2024-07-31 20:00:00,0.000
80642324,46.55,-103.85,2024-07-31,2024-07-31 19:00:00,0.000
80642322,46.55,-103.85,2024-07-31,2024-07-31 17:00:00,0.100


In [7]:
testing_df['h_precip_mm'].describe()

count    1.015171e+08
mean     2.573030e-01
std      1.240674e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.000000e-03
max      5.412500e+01
Name: h_precip_mm, dtype: float64